In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from lifelines import CoxPHFitter
import warnings

In [ ]:
# Suppress lifelines convergence warnings during stochastic search
warnings.filterwarnings("ignore")

In [ ]:
# 1. Define the extended set of powers P
POWER_SET = [None, -3, -2.5, -2, -1.5, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 1.5, 2, 2.5, 3]

In [ ]:
def fp_transform(x, p):
    """Applies a single fractional polynomial transformation."""
    if p is None:
        return None
    elif p == 0:
        return np.log(x)
    else:
        return x ** p

In [ ]:
def generate_fp_features(df, features, powers):
    """
    Generates a new dataframe with FP transformed features.
    powers is a list of tuples for each feature: [(p1_1, p2_1), (p1_2, p2_2), ...]
    """
    transformed_data = pd.DataFrame(index=df.index)
    
    for i, col in enumerate(features):
        x = df[col].astype(float)
        
        # Preliminary shifting: ensure positivity for log and fractional powers
        if (x <= 0).any():
            x = x - x.min() + 1e-5 
            
        p1, p2 = powers[i]
        
        valid_powers = [p for p in (p1, p2) if p is not None]
        valid_powers.sort()
        
        for j, p in enumerate(valid_powers):
            if j == 0:
                transformed_data[f"{col}_FP1_{p}"] = fp_transform(x, p)
            else:
                if p == valid_powers[j-1]:
                    transformed_data[f"{col}_FP2_rep"] = fp_transform(x, p) * np.log(x)
                else:
                    transformed_data[f"{col}_FP2_{p}"] = fp_transform(x, p)
                    
    return transformed_data

In [ ]:
def objective_function(de_vars, df, features, duration_col, event_col):
    """
    Objective function for Differential Evolution.
    Minimizes the BIC of the Cox Proportional Hazards model.
    """
    indices = np.floor(de_vars).astype(int)
    
    powers = []
    for i in range(len(features)):
        p1 = POWER_SET[indices[2*i]]
        p2 = POWER_SET[indices[2*i + 1]]
        powers.append((p1, p2))
        
    df_fp = generate_fp_features(df, features, powers)
    
    if df_fp.empty:
        return 1e10
        
    df_fp[duration_col] = df[duration_col]
    df_fp[event_col] = df[event_col]
    
    cph = CoxPHFitter(penalizer=0.01) 
    
    try:
        cph.fit(df_fp, duration_col=duration_col, event_col=event_col)
        
        n_events = df[event_col].sum()
        if n_events <= 1: 
            n_events = len(df)
            
        k = len(cph.params_)
        log_lik = cph.log_likelihood_
        
        bic = -2 * log_lik + k * np.log(n_events)
        return bic
        
    except Exception:
        return 1e10

In [ ]:
# --- NEW: Tracker class to capture the best BIC per iteration ---
class ConvergenceTracker:
    def __init__(self):
        self.best_val = np.inf
        self.history = []

    def evaluate(self, de_vars, df, features, duration_col, event_col):
        """Wraps the objective function to track the best seen value."""
        val = objective_function(de_vars, df, features, duration_col, event_col)
        if val < self.best_val:
            self.best_val = val
        return val

    def callback(self, xk, convergence=None):
        """Called by scipy.optimize at the end of every iteration."""
        # Append the best value found up to this iteration
        self.history.append(self.best_val)

In [ ]:
def select_fp_cox_de(df, features, duration_col, event_col, maxiter=50, popsize=15):
    """
    Runs Differential Evolution to find the best Fractional Polynomial powers for a Cox PH model.
    """
    dimensions = 2 * len(features)
    bounds = [(0, 15.999)] * dimensions
    
    print(f"Starting DE Optimization for {len(features)} features...")
    
    # Initialize the tracker
    tracker = ConvergenceTracker()
    
    result = differential_evolution(
        func=tracker.evaluate,
        bounds=bounds,
        args=(df, features, duration_col, event_col),
        maxiter=maxiter,
        popsize=popsize,
        mutation=(0.5, 1.0),
        recombination=0.7,
        seed=42,
        disp=True,
        callback=tracker.callback # Attach the callback
    )
    
    best_indices = np.floor(result.x).astype(int)
    best_powers = []
    
    print("\n--- Optimal Power Selection ---")
    for i in range(len(features)):
        p1 = POWER_SET[best_indices[2*i]]
        p2 = POWER_SET[best_indices[2*i + 1]]
        best_powers.append((p1, p2))
        print(f"{features[i]}: Power 1 = {p1}, Power 2 = {p2}")
        
    print(f"Minimum BIC Achieved: {result.fun}")
    
    # Plot the convergence
    plot_convergence(tracker.history)
    
    return best_powers, result.fun

In [ ]:
def plot_convergence(history):
    """Plots the convergence curve of the DE optimization."""
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(history) + 1), history, marker='o', linestyle='-', color='b')
    plt.title('Differential Evolution Convergence for FP Power Selection')
    plt.xlabel('Generation / Iteration')
    plt.ylabel('Best BIC Score')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# Example Usage with a Dummy Dataset
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)
    n = 300
    df = pd.DataFrame({
        'age': np.random.uniform(30, 80, n),
        'biomarker1': np.random.exponential(1.5, n),
        'biomarker2': np.random.normal(50, 10, n),
        'duration': np.random.randint(1, 100, n),
        'event': np.random.binomial(1, 0.7, n)
    })
    
    continuous_features = ['age', 'biomarker1', 'biomarker2']
    
    # Run the DE algorithm (you will see a plot pop up at the end)
    best_powers, best_bic = select_fp_cox_de(
        df=df, 
        features=continuous_features, 
        duration_col='duration', 
        event_col='event',
        maxiter=20,  
        popsize=10
    )
    
    final_df = generate_fp_features(df, continuous_features, best_powers)
    final_df['duration'] = df['duration']
    final_df['event'] = df['event']
    
    final_cph = CoxPHFitter(penalizer=0.01)
    final_cph.fit(final_df, duration_col='duration', event_col='event')
    print("\n--- Final Model Summary ---")
    final_cph.print_summary()